In [ ]:
import mne
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from mne.datasets.sleep_physionet.age import fetch_data

np.random.seed(42)

In [ ]:
# Data loading and feature extraction

stage_names = {
    'Wake': 'Sleep stage W',
    'N1':   'Sleep stage 1',
    'N2':   'Sleep stage 2',
    'N3':   'Sleep stage 3',
    'REM':  'Sleep stage R',
}

def extract_features(signal, fs):
    fft_result = np.fft.rfft(signal)
    powers     = np.abs(fft_result) ** 2
    freqs      = np.fft.rfftfreq(len(signal), d=1/fs)
    delta = np.sum(powers[(freqs >= 0.5) & (freqs < 4)])
    theta = np.sum(powers[(freqs >= 4)   & (freqs < 8)])
    alpha = np.sum(powers[(freqs >= 8)   & (freqs < 13)])
    beta  = np.sum(powers[(freqs >= 13)  & (freqs < 30)])
    theta_beta  = theta / (beta  + 1e-6)
    delta_theta = delta / (theta + 1e-6)
    return [delta, theta, alpha, beta, theta_beta, delta_theta]

all_X = []
all_y = []

for subject_id in [0, 1, 2, 3, 4]:
    files    = fetch_data(subjects=[subject_id], recording=[1])
    psg_file = files[0][0]
    hyp_file = files[0][1]
    raw         = mne.io.read_raw_edf(psg_file, preload=True, verbose=False)
    fs_real     = int(raw.info['sfreq'])
    data, _     = raw['EEG Fpz-Cz']
    annotations = mne.read_annotations(hyp_file)

    for stage_label, stage_raw in stage_names.items():
        for j in range(len(annotations)):
            if annotations.description[j] == stage_raw:
                onset     = annotations.onset[j]
                start_idx = int(onset * fs_real)
                end_idx   = int((onset + 30) * fs_real)
                if end_idx <= data.shape[1]:
                    epoch    = data[0, start_idx:end_idx]
                    features = extract_features(epoch, fs_real)
                    all_X.append(features)
                    all_y.append(stage_label)

X_real = np.array(all_X)
y_real = np.array(all_y)

print(f"X shape: {X_real.shape}")
print(f"Label distribution: { {s: np.sum(y_real==s) for s in stage_names} }")

In [ ]:
# Preprocessing and model training

X_train, X_test, y_train, y_test = train_test_split(
    X_real, y_real, test_size=0.2, random_state=42, stratify=y_real
)

scaler        = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# KNN
knn_grid = GridSearchCV(
    KNeighborsClassifier(),
    {'n_neighbors': [1, 3, 5, 7, 9, 11]},
    cv=cv, scoring='accuracy'
)
knn_grid.fit(X_train_scaled, y_train)
y_pred_knn = knn_grid.predict(X_test_scaled)

# SVM
svm_grid = GridSearchCV(
    SVC(kernel='rbf', class_weight='balanced', random_state=42),
    {'C': [0.1, 1, 10], 'gamma': [0.01, 0.1, 1]},
    cv=cv, scoring='accuracy'
)
svm_grid.fit(X_train_scaled, y_train)
y_pred_svm = svm_grid.predict(X_test_scaled)

# Random Forest
rf_grid = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42),
    {'n_estimators': [100, 200], 'max_depth': [None, 10, 20]},
    cv=cv, scoring='accuracy'
)
rf_grid.fit(X_train_scaled, y_train)
y_pred_rf = rf_grid.predict(X_test_scaled)

In [ ]:
# Evaluation

print("Overall Accuracy")
print(f"KNN: {accuracy_score(y_test, y_pred_knn):.2%}")
print(f"SVM: {accuracy_score(y_test, y_pred_svm):.2%}")
print(f"RF:  {accuracy_score(y_test, y_pred_rf):.2%}")

print("\nRF Classification Report")
print(classification_report(y_test, y_pred_rf, target_names=list(stage_names.keys())))